# **Método de Kasiski. Ejercicios para practicar.**

### **Ejercicio 1** ###

Al ejecutar la casilla siguiente, se carga un texto cifrado en la variable $\texttt{encriptado}$.

In [1]:
load('VetustaEnc.py')
len(encriptado)

73134

Dicho texto se corresponde con un capítulo de un libro de un escritor al que le gustaba especialmente la letra "a", quizá porque su primer apellido sólo tenía esa vocal. El texto encriptado se obtuvo a partir del original mediante un método de Vigenère, con el alfabeto que se proporciona en la siguiente casilla.

In [2]:
alfabeto="!'(),-.1234:;?abcdefghijklmnopqrstuvwxyz¡«º»¿ñ"

Dicho texto se corresponde con un capítulo de un libro de un escritor al que le gustaba especialmente la letra "a", quizá porque su primer apellido sólo tenía esa vocal. El texto encriptado se obtuvo a partir del original mediante un método de Vigenère, con el alfabeto que se proporciona en la siguiente casilla.

-------------------------------------

**Solución.** Hacemos un análisis de Kasiski para intentar encontrar la longitud de la clave. Para ello echamos mano de los programas de la sesión.

In [3]:
def ngramas(texto,n,m=30):
    diccio=dict()
    for k in xsrange(len(texto)-n+1):            # k es la posición de la letra de comienzo del n-grama
        ngrama=texto[k:k+n]                      # Este es el n-grama que comienza en la posición k 
        if ngrama not in diccio: diccio[ngrama]=1
        else: diccio[ngrama]+=1 
    lista=[(diccio[ngrama], ngrama) for ngrama in diccio]
    lista.sort(reverse=true)
    lista2=[(ngr,frec) for frec,ngr in lista]
    return lista2[:m]

def MCDdistancias(texto,ngrama):
    posiciones=[]
    for k in xsrange(len(texto)-len(ngrama)+1):  
        if texto[k:k+len(ngrama)]==ngrama: # Indentifico en qué posiciones k aparece el n-grama
            posiciones.append(k)           # Las añado a la lista posiciones
    
    # Solo tiene sentido calcular distancias si el n-grama aparece al menos dos veces
    if len(posiciones)>1: return gcd([posiciones[j+1] - posiciones[j] for j in xsrange(len(posiciones)-1)])
    else: return 0   # Si no hay al menos dos apariciones el programa devolverá 0 

Probamos con n-gramas de longitud 5.

In [4]:
distancias=[]
ngramasfrec=[ngr[0] for ngr in ngramas(encriptado,5)]
for ngrama in ngramasfrec:
    distancias.append(MCDdistancias(encriptado,ngrama))
print(distancias)

[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]


Parece sensato apostar por el 7.

In [5]:
lc=7 # Esta es la longitud de la clave obtenida mediante el análisis de Kasiski

Ya tenemos una apuesta de longitud de clave: $7$. Ahora solo hay que repetir lo que hicimos en la sesión anterior para lo que uso los programas que construimos entonces. Lo primero que hago es cargarlos.

In [6]:
#Funciones que hemos usado en la sesión de Vigenere. Las cargamos para usarlas

def contraVigenere(clave,alfabeto):
    
    # Compruebo si la clave es válida (todas sus letras están en el alfabeto)
    if any([letra not in alfabeto for letra in clave]): return 'La clave y el alfabeto son incompatibles.'
    
    # Traduzco la clave a números
    clavenum=[alfabeto.index(letra) for letra in clave]
    
    return ''.join([alfabeto[-k] for k in clavenum]) # Construyo la contraclave y la devuelvo

def cifradoVigenere(texto,clave,alfabeto):
    
    # Compruebo si la clave es válida (todas sus letras están en el alfabeto)
    if any([letra not in alfabeto for letra in clave]): return 'La clave y el alfabeto son incompatibles.'

    L=len(alfabeto) # longitud alfabeto
    long=len(clave) # longitud clave
    clavenum=[alfabeto.index(letra) for letra in clave] # Obtengo la versión numérica de la clave
    
    # Paso al cifrado propiamente dicho
    textocif=''
    for k in xsrange(len(texto)):
        letra=texto[k]
        if letra in alfabeto:
            letra=alfabeto[(alfabeto.index(letra)+clavenum[k%long])%L]           
        textocif+=letra
    return textocif

    
def caracteres(texto):
    lista=list(set(texto)) # Usamos set para eliminar repeticiones
    lista.sort()           # Lista ordenada, de menor a mayor
    return ''.join(lista)


def masfrecuentes(long,texto,alfabeto):
    palabra=''
    for j in xsrange(long):
        letras=texto[j::long]                                     # Letras en posiciones congruentes con j módulo long
        letrasUA=set(letras).intersection(alfabeto)               # Estas son las letras que se han usado y que están en el alfabeto
        pares=[(letras.count(letra),letra) for letra in letrasUA] # Cuento cuántas veces aparece cada una de las letras usadas del alfabeto
        pares.sort(reverse=true)
        palabra+=pares[0][1]
    return palabra


def contraclave_Vigenere(texto,alfabeto,palabra):
    long=len(palabra) # longitud de la clave
    palabracodif=masfrecuentes(long,texto,alfabeto) # palabra formada por la letra más frecuente de cada bloque en el texto codificadoe
    contra=[]         # lista en la que almacenaremos los índices de las letras de la contraclave
    for j in xsrange(long):
        nuevaposicion_masfrec=alfabeto.index(palabracodif[j]) # índice de la letra más frecuente en el bloque j en el texto codificado
        posicion_masfrec=alfabeto.index(palabra[j])           # índice de la letra más frecuente en el bloque j en el texto original
        contra.append(alfabeto[(-nuevaposicion_masfrec+posicion_masfrec)]) # letra de la contraclave correspondiente a la posición j 
    return ''.join(contra)                                    # junto las letras para formar la contraclave 

Como me han dicho cuál es el alfabeto, esa parte me la puedo saltar. Paso directamente a intentar obtener la contraclave, atendiendo a la pista de que al escritor le gustaba mucho la letra a.

In [7]:
contraclave=contraclave_Vigenere(encriptado,alfabeto,7*'a')

<p>Una vez conocemos la contraclave, simplemente tenemos que aplicar el programa de cifrado de Vigen&egrave;re al texto cifrado usando como clave la contraclave.</p>

Veamos si ha habido suerte.

In [8]:
original=cifradoVigenere(encriptado,contraclave,alfabeto)
print(original[:200])

la heroica ciudad dormia la siesta. el viento sur, caliente y perezoso,
empujaba las nubes blanquecinas que se rasgaban al correr hacia el
norte. en las calles no habia mas ruido que el rumor estriden


**¡Bingo!** Se trata de *La Regenta*, de Leopoldo Alas (Clarín).

In [9]:
# Veamos cuál era la clave

clave=contraVigenere(contraclave,alfabeto)
print(clave)

vetusta


----------------------------

### **Ejercicio 2** ###

Al ejecutar la casilla siguiente, se carga un texto cifrado en la variable $\texttt{encriptado2}$, resultado de encriptar mediante el método de Vigenère un texto escrito en español, sin tildes y todo en letras mayúsculas, con el alfabeto $\texttt{alfabeto2}$ proporcionado en esa misma casilla. Intenta averiguar el texto original.

In [10]:
load('HalconEnc.py')
alfabeto2='!,-.03458:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ¡«»¿Ñ—'
len(encriptado2)

24065

------------------------------

**Solución.** Seguimos la misma estrategia que en el ejercicio 1. Primero intentamos averiguar la longitud de la clave mediante un análisis de Kasiski.

In [11]:
# Pruebo con n-gramas de longitud 5
distancias=[]
ngramasfrec=[ngr[0] for ngr in ngramas(encriptado2,5)]
for ngrama in ngramasfrec:
    distancias.append(MCDdistancias(encriptado2,ngrama))
print(distancias)

[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 7, 7, 7]


In [12]:
lc=7 # Esta es la longitud de la clave obtenida mediante el análisis de Kasiski

In [13]:
contraclave2=contraclave_Vigenere(encriptado2,alfabeto2,7*'E')
original2=cifradoVigenere(encriptado2,contraclave2,alfabeto2)
print(original2[:200])

MUERXE EN LE NIEBLE  EN LE OSCURMDAD SORO EL TMMBRE DI UN TEPEFONO. DESPUEW DE QUI HUBO WONADO XRES VEGES, SE OYO EL CHIRRIHO DE LSS MUELPES DE YNA CAME; UNOS DEDOS TALPAROR SOBRE LA MADIRA, ALKO PEQU


Parece que nos estamos equivocando en alguna posición. Veamos cuáles.

In [14]:
print(original2[:7])

MUERXE 


Parece que nos estamos equivocando en las posiciones correspondientes al índice 4. Suponemos entonces que la letra más frecuente en esas posiciones en el original no era la E sino la A.

In [15]:
contraclave2=contraclave_Vigenere(encriptado2,alfabeto2,4*'E'+'A'+2*'E')
original2=cifradoVigenere(encriptado2,contraclave2,alfabeto2)
print(original2[:200])

MUERTE EN LA NIEBLA  EN LA OSCURIDAD SONO EL TIMBRE DE UN TELEFONO. DESPUES DE QUE HUBO SONADO TRES VECES, SE OYO EL CHIRRIDO DE LOS MUELLES DE UNA CAMA; UNOS DEDOS PALPARON SOBRE LA MADERA, ALGO PEQU


Ahora está todo bien. Para terminar buscamos la clave con la que se codificó el texto. 

In [16]:
clave2=contraVigenere(contraclave2,alfabeto2)
print(clave2)

HALCONX


Se trata de "El halcón maltés", de Dashiell Hammett. Hay una famosa versión cinematográfica del director John Huston.